# CIFAR-10 baseline vs looped embeddings

Default looped checkpoint trails baseline by −2.1 pp (tuned probe). Uses `load_ijepa` and figure code under `visualizations/`.

Pre-built figures (regenerate: `visualizations/generate_all_figures.py`):
- [`../visualizations/figures/03_embeddings.png`](../visualizations/figures/03_embeddings.png)
- [`../visualizations/figures/03_per_loop_cosine.png`](../visualizations/figures/03_per_loop_cosine.png)
- [`../visualizations/loop_analysis/01_exit_distribution.png`](../visualizations/loop_analysis/01_exit_distribution.png)

Summary: [`FINDINGS.md`](FINDINGS.md)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "jepa").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10

from jepa import load_ijepa
from jepa.data.cifar10 import build_dataloaders
from jepa.eval.linear_probe import extract_features, train_probe_head
from jepa.masking import IJEPAMaskCollator
from jepa.utils.device import get_device
from jepa.utils.seed import set_seed
from jepa.utils.weights import get_released_weight

from visualizations.style import apply_style
from visualizations.figures.embeddings import plot_embedding_comparison, plot_per_loop_cosine
from visualizations.figures.loop_analysis import plot_exit_distribution_deep, plot_cosine_and_l1_by_loop
from visualizations.loop_collect import collect_loop_sample_records
from notebooks.latent_metrics import (
    compare_separation,
    nearest_centroid_confusion_pairs,
    plot_feat_std_panel,
    per_class_probe_disagreements,
    plot_disagreement_tiles,
)

CIFAR10_CLASSES = list(CIFAR10(root="data", train=False, download=True).classes)
apply_style()
set_seed(42)
device = get_device("auto")
NOTEBOOK_OUT = PROJECT_ROOT / "notebooks" / "outputs"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)
print(f"device={device}  root={PROJECT_ROOT}")

## 1. Load released checkpoints

Uses `load_ijepa` (local `checkpoints/` or download when URLs are live). Sandwich is optional.

In [ ]:
baseline = load_ijepa("baseline_v3", pretrained=True, device=device)
looped = load_ijepa("looped_v3", pretrained=True, device=device)

sandwich = None
sandwich_spec = get_released_weight("sandwich_rmsnorm")
if (PROJECT_ROOT / sandwich_spec.checkpoint).is_file():
    sandwich = load_ijepa("sandwich_rmsnorm", pretrained=True, device=device)
    print(f"sandwich checkpoint: {sandwich_spec.checkpoint}")
else:
    print("sandwich_rmsnorm checkpoint not found — skipping optional third model")

for name, m in [("baseline", baseline), ("looped", looped)]:
    print(f"{name}: trainable params={m.num_trainable_params():,}")

## 2. Frozen-encoder embeddings (t-SNE / UMAP)

Mean-pooled patch tokens on the full CIFAR-10 val split. Side-by-side projection; pre-rendered figure linked below.

In [ ]:
_, val_loader = build_dataloaders(batch_size=128, train_augment=False, num_workers=0)
base_feats, labels = extract_features(baseline, val_loader, device)
loop_feats, labels2 = extract_features(looped, val_loader, device)
assert torch.equal(labels, labels2)

plot_embedding_comparison(
    base_feats, loop_feats, labels,
    output_stem=NOTEBOOK_OUT / "embeddings_tsne",
    max_points=2000,
    method="tsne",
)

# Optional UMAP (falls back to skipping if umap-learn not installed)
try:
    from visualizations.figures.embeddings import _project_2d
    import umap
    import matplotlib.pyplot as plt
    from visualizations.style import save_figure

    def _umap_2d(feats):
        reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
        return reducer.fit_transform(feats.detach().cpu().numpy())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    for ax, proj_fn, title, feats in [
        (axes[0], _umap_2d, "baseline (UMAP)", base_feats),
        (axes[1], _umap_2d, "looped (UMAP)", loop_feats),
    ]:
        xy = proj_fn(feats)
        ax.scatter(xy[:, 0], xy[:, 1], c=labels.cpu(), cmap="tab10", s=8, alpha=0.75)
        ax.set_title(title)
    fig.suptitle("Frozen encoder features (UMAP)")
    fig.tight_layout()
    save_figure(fig, NOTEBOOK_OUT / "embeddings_umap")
    print("UMAP panel saved")
except ImportError:
    print("umap-learn not installed — t-SNE panel only (see viz extra or pip install umap-learn)")

from IPython.display import Image, display
prebuilt = PROJECT_ROOT / "visualizations" / "figures" / "03_embeddings.png"
if prebuilt.is_file():
    display(Image(filename=str(prebuilt)))
else:
    display(Image(filename=str(NOTEBOOK_OUT / "embeddings_tsne.png")))

## 3. Quantitative separation

Silhouette score and nearest-centroid accuracy on frozen features. Worst class pairs are read from the **data**, not guessed.

In [ ]:
sep = compare_separation(base_feats, loop_feats, labels)
print("Silhouette / nearest-centroid accuracy:")
for k, m in sep.items():
    print(f"  {k:8}  silhouette={m.silhouette:.4f}  nearest_centroid_acc={m.nearest_centroid_acc*100:.2f}%")

print("\nWorst nearest-centroid pairs (baseline):")
for true_c, pred_c, rate, count in nearest_centroid_confusion_pairs(base_feats, labels, CIFAR10_CLASSES, top_k=6):
    print(f"  {true_c:12} -> {pred_c:12}  rate={rate*100:5.1f}%  n={count}")

print("\nWorst nearest-centroid pairs (looped):")
for true_c, pred_c, rate, count in nearest_centroid_confusion_pairs(loop_feats, labels, CIFAR10_CLASSES, top_k=6):
    print(f"  {true_c:12} -> {pred_c:12}  rate={rate*100:5.1f}%  n={count}")

## 4. Predictor-space: per-loop cosine to teacher targets

Reuses `plot_per_loop_cosine` and per-class loop analysis from `visualizations/figures/loop_analysis.py`.

In [ ]:
from jepa.utils.config import load_config

cfg = load_config(PROJECT_ROOT / "configs" / "image_jepa_cifar10_v3_looped.yaml")
grid_size = cfg["data"]["img_size"] // cfg["data"]["patch_size"]
collator = IJEPAMaskCollator(
    grid_size=grid_size,
    fixed_context_patches=cfg["masking"].get("fixed_context_patches", 32),
    fixed_target_patches=cfg["masking"].get("fixed_target_patches", 16),
)

plot_per_loop_cosine(looped, val_loader, collator, device, NOTEBOOK_OUT / "per_loop_cosine", max_batches=32)

records, _ = collect_loop_sample_records(looped, val_loader, collator, device, max_batches=16)
plot_cosine_and_l1_by_loop(records, CIFAR10_CLASSES, NOTEBOOK_OUT / "cosine_l1_by_loop")

prebuilt = PROJECT_ROOT / "visualizations" / "figures" / "03_per_loop_cosine.png"
if prebuilt.is_file():
    display(Image(filename=str(prebuilt)))

## 5. Feature norm distribution (`feat_std`)

Registry `feat_std` is mean per-dimension std **after train-split standardization** in the tuned probe. Here we plot **raw L2 norms** to separate collapse from tighter scaling.

Sandwich-RMSNorm: `feat_std=0.0432` **and** best probe (78.28%) → compressed norms, not dead features.

In [ ]:
feat_dict = {"baseline_v3": base_feats, "looped_v3": loop_feats}
ref = {
    "baseline_v3": get_released_weight("baseline_v3").feat_std,
    "looped_v3": get_released_weight("looped_v3").feat_std,
}
if sandwich is not None:
    sand_feats, _ = extract_features(sandwich, val_loader, device)
    feat_dict["sandwich_rmsnorm"] = sand_feats
    ref["sandwich_rmsnorm"] = get_released_weight("sandwich_rmsnorm").feat_std

plot_feat_std_panel(feat_dict, NOTEBOOK_OUT / "feat_std_panel", reference_feat_std=ref)
display(Image(filename=str(NOTEBOOK_OUT / "feat_std_panel.png")))

## 6. Exit-gate behaviour

Distribution of expected loops and mean P(exit) per loop index.

In [ ]:
plot_exit_distribution_deep(records, NOTEBOOK_OUT / "exit_distribution")

prebuilt = PROJECT_ROOT / "visualizations" / "loop_analysis" / "01_exit_distribution.png"
if prebuilt.is_file():
    display(Image(filename=str(prebuilt)))

mean_loops = sum(r.expected_loops for r in records) / max(1, len(records))
print(f"mean expected loops={mean_loops:.2f}  (gate ~uniform on this checkpoint)")

## 7. Qualitative probe disagreements

Train lightweight linear probes in-memory (no saved probe checkpoint). Show tiles where looped is wrong and baseline is right, and the reverse.

In [ ]:
train_loader, probe_val_loader = build_dataloaders(batch_size=256, train_augment=False, num_workers=0)
embed_dim = cfg["encoder"]["embed_dim"]
b_head = train_probe_head(baseline, train_loader, probe_val_loader, device, embed_dim, epochs=20)
l_head = train_probe_head(looped, train_loader, probe_val_loader, device, embed_dim, epochs=20)

p_base, p_labels = extract_features(baseline, probe_val_loader, device)
p_loop, _ = extract_features(looped, probe_val_loader, device)

dis = per_class_probe_disagreements(p_base, p_loop, p_labels, b_head, l_head, device, n_each=6)

# Collect raw images for tile display
val_ds = probe_val_loader.dataset
images_list, labels_list = [], []
for img, y in DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0):
    images_list.append(img)
    labels_list.append(y)
images = torch.cat(images_list)
labels_all = torch.cat(labels_list)

with torch.no_grad():
    b_pred = b_head(p_base.to(device)).argmax(1).cpu()
    l_pred = l_head(p_loop.to(device)).argmax(1).cpu()

plot_disagreement_tiles(
    images, labels_all, CIFAR10_CLASSES,
    dis["looped_wrong_baseline_right"],
    "Looped wrong / baseline right",
    b_pred, l_pred,
    NOTEBOOK_OUT / "tiles_looped_wrong",
)
plot_disagreement_tiles(
    images, labels_all, CIFAR10_CLASSES,
    dis["baseline_wrong_looped_right"],
    "Baseline wrong / looped right",
    b_pred, l_pred,
    NOTEBOOK_OUT / "tiles_baseline_wrong",
)
display(Image(filename=str(NOTEBOOK_OUT / "tiles_looped_wrong.png")))
display(Image(filename=str(NOTEBOOK_OUT / "tiles_baseline_wrong.png")))

## Summary

See [`FINDINGS.md`](FINDINGS.md).